# Python — Pydantic, Dataclasses, and the Type System
**Day 10 — Python Module**

---

<div style="padding:15px;border-left:8px solid #667eea;background:#f0f0ff;border-radius:4px;">
<strong>Core Insight:</strong> Data pipelines fail because of bad data. Pydantic validates data at the boundary — when data enters your system. Type hints + Pydantic catch schema violations, wrong types, and missing fields at intake, not three steps later when a KeyError crashes production.
</div>

### Why It Matters
Relying purely on python's native dynamic typing leads to fragile downstream logic (e.g., executing `.upper()` on an integer that slipped cleanly through an API payload). Establishing hard constraints via Dataclasses internally, and Pydantic schema validation externally, fundamentally locks out poisoned data.

## Python Type Guards & Model Schemas

```python
# Dataclasses (Internal Logic) -> Trust-Based Structs
from dataclasses import dataclass, field
from typing import Optional, List

@dataclass
class MetricRecord:
    cpu_pct: float
    tags: List[str] = field(default_factory=list) # Mutable protection!

# Pydantic (External Logic) -> Zero-Trust Validation Arrays
from pydantic import BaseModel, Field

class MetricSchema(BaseModel):
    server_id: str = Field(pattern=r"^srv-")
    cpu_pct: float = Field(ge=0.0, le=100.0)      # Strict threshold validation
```

**Rule:** Use `dataclasses` inside your application when functions pass data to other functions you wrote. Use `Pydantic` the exact second data crosses over a boundary interacting with your domain (e.g. from JSON payloads, Database rows, Kafka strings, API calls).

In [ ]:
# Runnable Demo: Dataclasses vs Raw Dicts
from dataclasses import dataclass, field
from datetime import date

@dataclass
class ServerMetric:
    server_id: str
    report_date: date
    avg_cpu: float
    region: str
    tier: str = "silver"           # default value
    alert_threshold: float = 90.0
    tags: list[str] = field(default_factory=list)  # vital mutable default

    def is_critical(self) -> bool:
        return self.avg_cpu >= self.alert_threshold

    def __post_init__(self):
        """Executes safely and strictly after standard __init__."""
        if not (0 <= self.avg_cpu <= 100):
            raise ValueError(f"avg_cpu must be 0-100, got {self.avg_cpu}")
        if self.server_id.strip() == "":
            raise ValueError("server_id cannot be empty")

# Executing comparative tests on safety
print("Testing Raw Dict:")
raw_payload = {"server_id": "srv-01", "cpu_perc": 85.3}
try:
    alert = raw_payload['cpu'] > 90.0
except KeyError as e:
    print(f"Raw Dict failed on parsing logic later due to KeyError: {e}\n")

print("Testing Dataclass init:")
m = ServerMetric(server_id="srv-01", report_date=date(2026, 2, 26), avg_cpu=85.3, region="us-east")
print(f"Dataclass Created: {m}")
print(f"Is threshold critical? {m.is_critical()}")

## BaseModels & Pydantic Validation

Pydantic forces Python to be strict when ingesting JSON bytes.

**Execution Mechanics:**
1. Pydantic accepts `kwargs` into a BaseModel.
2. It cross-checks incoming strings against the configured fields (e.g. `Field(ge=0)` check guarantees value applies logically).
3. **Coercion occurs:** If an API passes `"avg_cpu": "72"` (a string), Pydantic forcibly coerces it safely into the explicit `float` typing block.
4. Custom validators run sequentially.

In [ ]:
# Runnable Demo: Pydantic Boundaries handling Citi-Adjacent payloads
from pydantic import BaseModel, Field, field_validator, ValidationError
from datetime import date
from typing import Literal

class ServerMetricSchema(BaseModel):
    # Validation configurations inside defining model
    server_id: str = Field(min_length=1, pattern=r"^srv-\d{2}$")
    report_date: date
    avg_cpu: float = Field(ge=0.0, le=100.0)   # >= 0, <= 100
    region: Literal['us-east', 'us-west', 'eu-west']
    tier: Literal['gold', 'silver', 'bronze'] = 'silver'

    @field_validator('server_id')
    @classmethod
    def server_id_uppercase(cls, v: str) -> str:
        return v.lower()   # normalize bad intake formatting automatically

# GOOD data intake: (Including the Auto-Coercion on string -> date)
print("Good Data Validates Perfectly (Notice type coercion on report_date string -> Date):")
good_m = ServerMetricSchema(
    server_id="SRV-01",         # Will hit standard normalizer
    report_date="2026-02-26",   # Implicit coercion to formal Date
    avg_cpu=85.3,
    region="us-east",
    tier="gold"
)
print(good_m.model_dump())

# BAD data intake: Raises structured exceptions
print("\nBad Data Fails Safe During Intake:")
try:
    bad = ServerMetricSchema(
        server_id="",               # fails min_length
        report_date="2026-02-26",
        avg_cpu=150.0,              # fails le=100
        region="eu-north",          # not in Literal definition
    )
except Exception as e:
    print(f"Validation failed: {e}")

In [ ]:
# Pipeline Showcase — Processing streams error-free
import json
from pydantic import ValidationError
from typing import Optional

def process_incoming_event(raw_json: str) -> Optional[ServerMetricSchema]:
    """Parse and validate an incoming telemetry event perfectly at the boundary."""
    try:
        data = json.loads(raw_json)
        metric = ServerMetricSchema(**data)
        return metric
    except (json.JSONDecodeError, ValidationError) as e:
        # Log to dead-letter queue, don't crash the pipeline iterator
        print(f"[REJECTED] Invalid event: {e}")
        return None

# Batch intake array containing valid, invalid values, and pure garbage JSON bytes
batch_data = [
    '{"server_id": "srv-01", "report_date": "2026-02-27", "avg_cpu": 50.0, "region": "us-east"}',
    '{"server_id": "srv-02", "report_date": "2026-02-27", "avg_cpu": 120.0, "region": "us-west"}',
    'BAD_JSON_GIBBERISH'
]

def ingest_batch(events: list[str]) -> tuple[list, list]:
    valid, invalid = [], []
    for event in events:
        result = process_incoming_event(event)
        if result:
            valid.append(result)
        else:
            invalid.append(event)
    print(f"Ingested: {len(valid)} valid | {len(invalid)} rejected")
    return valid, invalid

v, iv = ingest_batch(batch_data)
print(f"Valid Array contains object -> {v[0].model_dump()}")

## Integrating Type System Protections

Pydantic natively leverages Python's modern typing system (like `dict`, `list`, `Optional`) to compile validation functions directly into highly optimized Rust under the hood natively providing massive telemetry parsing speeds.

In [ ]:
# Fast library integration functions utilizing Pydantic Types
from typing import List, Optional
from pydantic import BaseModel

class LocalConfig(BaseModel):
    server_id: str
    region: str
    status_codes: List[int] = [200, 404, 500]  # Pydantic safely constructs individual Mutables
    fallback_flag: Optional[bool] = None

# 1. Export formatting using .model_dump_json()
valid_instance = LocalConfig(server_id='localhost', region='us-east', status_codes=[200, 404])
print("JSON formatted output ready for export:", valid_instance.model_dump_json())

# 2. Catching completely isolated omissions
try:
    flawed_instance = LocalConfig(**{'server_id': 'localhost'})
except Exception as e:
    print(f"Failed configuration missing keys: {e}")

# 3. Unpacking valid dicts easily
print(f"Extracted values cleanly -> Server {valid_instance.server_id.capitalize()} resides in {valid_instance.region.upper()}.")

## Interview Q&A

**Q: Pydantic vs dataclasses — when do you use each?**<br>
A: Dataclasses: internal data containers where you explicitly trust the data source (Python code calling Python code). Pydantic: external data boundaries — API responses, frontend user inputs, JSON logs from Kafka, or explicit database rows. Pydantic validates and coerces; Dataclasses purely implicitly trust.

**Q: What is `field(default_factory=list)` and why is it definitively required?**<br>
A: Mutable defaults (like `list` or `dict`) cannot be shared across physical python instances. Supplying `default_factory` forces Python to create a totally blank slate, fresh instance correctly. Defining `= []` physically inside parameters is a known catastrophic Python bug directly copying memory refs.

**Q: What exactly does `model_dump()` do?**<br>
A: It systematically unwinds the Pydantic instance into a native python `dict`. Alternatively, executing `model_dump_json()` bypasses intermediate processing and securely coerces everything into formatted JSON string bytes directly representing the exact inverse function of the constructor.

**Q: What is the dead-letter queue pattern?**<br>
A: Instead of crashing outright on corrupt payload strings, you isolate and route the toxic records to a monitored "dead-letter" location (e.g. S3 path, separate DB table). This allows massive analytical pipelines to proceed continuously and safely unbothered, while preserving toxic logs for later debug replays.

**Q: If you specify `age: int` inside a Pydantic model and someone passes `'age': '42'` in their JSON, what happens?**<br>
A: Pydantic successfully auto-coerces it. Unlike standard static type checking, Pydantic attempts to fulfill the type-contract via `int('42')`. If the user passes `'age': 'forty-two'`, it throws a permanent structured `ValidationError` blocking execution immediately before entering application logic.

## The Citi Angle

**Securing Pipeline Aggregation at External Boundaries**

```python
from pydantic import BaseModel, Field, ValidationError
from datetime import datetime
from typing import Optional

class TelemetryCheck(BaseModel):
    server_identifier: str = Field(min_length=3)
    recorded_time: datetime
    cpu_metric: float = Field(ge=0, le=100)

def handle_message(message: dict) -> Optional[TelemetryCheck]:
    try:
        return TelemetryCheck(**message)
    except ValidationError as error_context:
        route_to_dlq(message, context=error_context)
        return None
```

*"One of the persistent problems at Citi was data quality in the telemetry pipeline. Agents would report metrics with wrong timestamps — sometimes future-dated by hours due to NTP drift. This would corrupt rolling averages and make dashboards show erratic capacity trends. I built a validation layer that checked timestamp sanity (reject if more than 30 minutes in the future or more than 24 hours old) and server ID validity against our inventory. What I'd do today with dbt: add schema tests on the raw source and custom data tests for timestamp bounds — catch it at the staging layer before it reaches the mart. The dead-letter queue would hold rejects for manual review."*